In [11]:
import pandas as pd
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def _parse_duration(text):
    if pd.isna(text): return 0
    h = re.search(r'(\d+)\s*hour', str(text))
    m = re.search(r'(\d+)\s*min', str(text))
    return (int(h.group(1)) if h else 0)*60 + (int(m.group(1)) if m else 0)

def _extract_genres(raw):
    genres = []
    for part in str(raw).split(','):
        match = re.match(r'#\d+\s+in\s+(.+)', part.strip())
        if match:
            g = match.group(1).strip()
            if 'Audible Audiobooks' not in g and 'See Top' not in g:
                genres.append(g)
    return genres

# Load and merge
df1 = pd.read_csv("data/Audible_Catlog.csv")
df2 = pd.read_csv("data/Audible_Catlog_Advanced_Features.csv")
df = pd.merge(df2, df1[['Book Name','Author']], on=['Book Name','Author'], how='left')

# Clean
df['Description'] = df['Description'].fillna('').str.strip()
df['Ranks and Genre'] = df['Ranks and Genre'].fillna('')
df['duration_mins'] = df['Listening Time'].apply(_parse_duration)
df['genres_list'] = df['Ranks and Genre'].apply(_extract_genres)
df['primary_genre'] = df['genres_list'].apply(lambda x: x[0] if x else 'Other')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce').fillna(0).astype(int)
df = df.sort_values('Number of Reviews', ascending=False).drop_duplicates('Book Name')

df.to_csv("data/books_clean.csv", index=False)
print("✅ Cleaned dataset saved")


✅ Cleaned dataset saved


In [12]:
df = pd.read_csv("data/books_clean.csv")

df_top = df.nlargest(600, 'Number of Reviews').reset_index(drop=True)
df_top['text_features'] = (
    df_top['Description'].str[:400] + ' ' + df_top['primary_genre'] + ' ' + df_top['Author']
)

vec = TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1,2))
mat = vec.fit_transform(df_top['text_features'].fillna(''))
sim = cosine_similarity(mat)

# Save pickle files
pickle.dump(vec, open("models/tfidf_vectorizer.pkl","wb"))
pickle.dump(sim, open("models/similarity_matrix.pkl","wb"))
df_top.to_csv("data/books_clustered.csv", index=False)

print("✅ Model built and pickle files saved")

✅ Model built and pickle files saved
